# Translating German to English inside Teradata with BYOM

This notebook walks through an end-to-end demonstration of running a
neural machine translation model **inside Teradata Vantage** using
the Bring Your Own Model (BYOM) framework. Starting from an
open-source HuggingFace checkpoint, you will export the model to
ONNX, deploy it into a Teradata database alongside its tokenizer,
and translate a set of German sentences to English with a single SQL
query — no Python sidecar, no data movement, no external inference
service.

The same pattern generalizes to any sequence-to-sequence model that
exports cleanly to ONNX with embedded beam search: translation,
summarization, code generation, instruction-following, and so on.


## Install

This notebook needs the `teradata-opus-translate` package (the model
converter you'll call below) and `teradataml` (the official Teradata
Python client used to upload BLOBs and query the database). Run the
cell below once per environment.

> **Note on outputs.** This notebook is committed with sanitised
> example outputs so you can read it top-to-bottom without a
> database connection. When you execute the cells against your own
> Teradata instance, the values you see (timings, server version,
> exact translations) will of course differ — the structure stays
> the same.


In [0]:
%pip install --quiet teradata-opus-translate teradataml


Note: you may need to restart the kernel to use updated packages.


## What is Teradata BYOM?

**Bring Your Own Model** is the Teradata pattern for scoring
externally trained machine learning models inside the database. You
train (or download) a model with the framework of your choice,
export it to a portable format, store the binary in a Teradata
table, and invoke it from SQL via a table operator that runs in
parallel across every AMP.

Why this matters:

- **Data gravity.** Production data already lives in the warehouse.
  Moving terabytes out to a Python service to score them is slow,
  expensive, and a governance headache. BYOM moves the model to the
  data instead.
- **Parallelism for free.** A single SQL invocation fans out across
  every AMP. Scoring a million rows looks the same as scoring eight
  — Teradata does the partitioning.
- **One environment, one audit trail.** Inputs, outputs, model
  binary, and access controls are all SQL objects in the same
  database. No separate model server to provision, monitor, or
  secure.

BYOM accepts models in PMML, H2O MOJO, and ONNX formats. This demo
uses **ONNX-Seq2Seq**, the BYOM operator that supports
encoder-decoder transformer models with embedded beam-search
generation.


## Why the OPUS-MT models?

The model used here is `Helsinki-NLP/opus-mt-de-en`, part of the
**OPUS-MT** family released by the Language Technology Research
Group at the University of Helsinki. The OPUS-MT collection covers
**hundreds of language pairs** — far more than any commercial
translation API — and the checkpoints are fully open-source under
permissive licenses.

Each model is a **MarianMT** encoder-decoder transformer with
roughly 75 million parameters. They are small, fast on CPU, and
produce strong translations for the languages they cover. Because
they are released as standard HuggingFace checkpoints, they slot
straight into the standard ONNX export tooling.

For this notebook we translate **German to English**, but the same
recipe applies unchanged to any other pair: French→English,
English→Spanish, Russian→French, and so on.


## Why translate inside the database?

Translation is a textbook example of an embarrassingly parallel,
row-wise workload over data that is already in the warehouse:
customer reviews, support tickets, product descriptions, financial
filings. The traditional answer — extract rows to a Python service,
translate, load results back — has three problems: latency from
network round-trips, cost from data movement at scale, and an
inference service to operate.

Running translation inside Teradata via BYOM eliminates all three:

- The model binary lives in a table; the tokenizer travels with it.
- A single SQL `SELECT` translates every row, in parallel, in place.
- Ten million rows or ten — same query, same code, no orchestration.

The rest of this notebook shows the full pipeline end-to-end.


## Setup

A small set of imports and the connection parameters for your
Teradata instance. The host, user, password, and target database
are read from environment variables so the notebook can be re-run
against any Teradata Vantage instance with `TD_MLDB.ONNXSeq2Seq`
available.


In [1]:
from __future__ import annotations

import os
import time
from pathlib import Path

import pandas as pd

# ----------------------------------------------------------------
# Connection parameters. Set these for your Teradata instance,
# either by editing the placeholders below or by exporting the
# matching environment variables before launching the notebook.
# The placeholder strings are intentionally invalid hostnames so
# the connection step fails loudly if the values are not set.
# ----------------------------------------------------------------
TD_HOST = os.environ.get("TD_HOST", "<your-teradata-host>")
TD_USER = os.environ.get("TD_USER", "<your-user>")
TD_PASSWORD = os.environ.get("TD_PASSWORD", "<your-password>")

# Database that holds the BYOM tables. Pick any database your
# user has CREATE TABLE rights on.
TD_BYOM_DATABASE = os.environ.get("TD_BYOM_DATABASE", "OPUS_BYOM")

# Model identity. Same string is the HuggingFace checkpoint name
# AND the row id we use inside Teradata's BYOM tables.
MODEL_ID = "Helsinki-NLP/opus-mt-de-en"
BYOM_MODEL_ID = "opus-mt-de-en"

# Local artifact cache. The exported ONNX model is ~743 MB so we
# keep it on disk between runs.
CACHE_DIR = Path.home() / ".cache" / "teradata-opus-translate"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
ONNX_PATH = CACHE_DIR / "de-en.onnx"
TOKENIZER_PATH = CACHE_DIR / "de-en.tokenizer.json"

print(f"Target Teradata host: {TD_HOST}")
print(f"BYOM database:        {TD_BYOM_DATABASE}")
print(f"Model:                {MODEL_ID}")


Target Teradata host: <your-teradata-host>
BYOM database:        OPUS_BYOM
Model:                Helsinki-NLP/opus-mt-de-en


## Step 1 — The input data

Six German sentences, chosen to exercise different sides of the
translation model: a casual greeting, a business sentence with
named numbers, a technical sentence, a German idiom, a sentence
with named entities and a date, and a long sentence with nested
subordinate clauses. These are the rows we will translate.


In [2]:
DEMO_SENTENCES: list[tuple[str, str]] = [
    ("greeting",  "Hallo Welt, wie geht es dir heute?"),
    ("business",  "Die Quartalszahlen werden nächste Woche um neun Uhr veröffentlicht."),
    ("technical", "Der neue Compiler optimiert den Maschinencode für moderne Prozessoren."),
    ("idiom",     "Ich verstehe nur Bahnhof."),
    ("entity",    "Albert Einstein wurde 1879 in Ulm geboren und entwickelte die Relativitätstheorie."),
    ("long",      "Obwohl es stark regnete, beschlossen wir, den Spaziergang im Park "
                  "fortzusetzen, weil das Wetter laut Vorhersage am Nachmittag besser werden sollte."),
]

inputs_df = pd.DataFrame(DEMO_SENTENCES, columns=["id", "source_de"])
inputs_df


,id,source_de
0,greeting,"Hallo Welt, wie geht es dir heute?"
1,business,Die Quartalszahlen werden nächste Woche um neu...
2,technical,Der neue Compiler optimiert den Maschinencode ...
3,idiom,Ich verstehe nur Bahnhof.
4,entity,Albert Einstein wurde 1879 in Ulm geboren und ...
5,long,"Obwohl es stark regnete, beschlossen wir, den ..."


## Step 2 — The model

Load `Helsinki-NLP/opus-mt-de-en` directly from the HuggingFace
hub. This is the same MarianMT encoder-decoder transformer that
will, in a few cells, be running inside Teradata. The cell below
also runs a quick sanity translation locally so you can see the
model produces sensible output before any database work begins.


In [3]:
import torch
from transformers import MarianMTModel, MarianTokenizer

torch.set_num_threads(1)  # deterministic CPU timing

print(f"Loading {MODEL_ID} ...")
hf_tokenizer = MarianTokenizer.from_pretrained(MODEL_ID)
hf_model = MarianMTModel.from_pretrained(MODEL_ID).eval()

n_params = sum(p.numel() for p in hf_model.parameters())
print(f"  parameters : {n_params:,} ({n_params / 1e6:.1f}M)")
print(f"  vocabulary : {hf_tokenizer.vocab_size:,} tokens")
print(f"  source     : German")
print(f"  target     : English")

# A single example translation as a 'look, it works' beat.
sample = "Hallo Welt."
enc = hf_tokenizer(sample, return_tensors="pt")
out_ids = hf_model.generate(**enc, num_beams=4, max_length=64)
print(f"\n  sample : {sample!r}")
print(f"  result : {hf_tokenizer.decode(out_ids[0], skip_special_tokens=True)!r}")


Loading Helsinki-NLP/opus-mt-de-en ...


/home/claude/workspace/projects/teradata-opus-translate/.venv/lib/python3.12/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


  parameters : 74,410,496 (74.4M)
  vocabulary : 58,101 tokens
  source     : German
  target     : English

  sample : 'Hallo Welt.'
  result : 'Hello, world.'


## Step 3 — Convert the model to ONNX

ONNX is a portable, framework-neutral format for neural networks.
Exporting the MarianMT model to ONNX produces a single
self-contained graph that bundles three pieces:

1. The **encoder** — turns source-language tokens into hidden states.
2. The **decoder** — generates target-language tokens autoregressively.
3. An embedded **beam-search loop** — the standard generation
   strategy for encoder-decoder MT, expressed as a single ONNX
   contrib operator (`com.microsoft.BeamSearch`).

Embedding beam search inside the graph is what makes the model
self-sufficient inside the database: Teradata's ONNX runtime sees
one node, calls it once per row, and gets back a sequence of
token ids. No external generation loop, no Python.

The export is a one-time, offline step. After this cell runs, the
743 MB ONNX file is the only artifact you need.


In [4]:
# The conversion bundles encoder + decoder + BeamSearch into a
# single ONNX file with the exact I/O shape Teradata's
# ONNXSeq2Seq operator expects.
from teradata_opus_translate import convert_model

if ONNX_PATH.exists():
    size_mb = ONNX_PATH.stat().st_size / (1024 * 1024)
    print(f"ONNX model already exported at {ONNX_PATH} ({size_mb:.1f} MB)")
else:
    print(f"Exporting {MODEL_ID} to ONNX ...")
    t0 = time.perf_counter()
    convert_model(MODEL_ID, output_path=ONNX_PATH, verify=True)
    elapsed = time.perf_counter() - t0
    size_mb = ONNX_PATH.stat().st_size / (1024 * 1024)
    print(f"Done in {elapsed:.1f}s — {size_mb:.1f} MB at {ONNX_PATH}")

print(f"ONNX file size: {ONNX_PATH.stat().st_size:,} bytes")


ONNX model already exported at /home/claude/.cache/teradata-opus-translate/de-en.onnx (709.2 MB)
ONNX file size: 743,632,928 bytes


## Step 4 — Extract the tokenizer

A model on its own cannot translate strings — it operates on token
ids. The tokenizer encodes input strings into ids and decodes output
ids back into strings. For BYOM it travels alongside the model as a
single `tokenizer.json` file, which Teradata loads server-side so
the entire encode → infer → decode pipeline happens in the database.


In [5]:
from teradata_opus_translate import convert_tokenizer

if TOKENIZER_PATH.exists():
    size_kb = TOKENIZER_PATH.stat().st_size / 1024
    print(f"Tokenizer already extracted at {TOKENIZER_PATH} ({size_kb:.1f} KiB)")
else:
    print(f"Extracting tokenizer for {MODEL_ID} ...")
    convert_tokenizer(MODEL_ID, output_path=TOKENIZER_PATH)
    size_kb = TOKENIZER_PATH.stat().st_size / 1024
    print(f"Done — {size_kb:.1f} KiB at {TOKENIZER_PATH}")

print(f"Tokenizer file size: {TOKENIZER_PATH.stat().st_size:,} bytes")


Tokenizer already extracted at /home/claude/.cache/teradata-opus-translate/de-en.tokenizer.json (3670.6 KiB)
Tokenizer file size: 3,758,710 bytes


## Step 5 — Deploy the model and tokenizer to Teradata

BYOM stores model artifacts as `BLOB` rows in simple two-column
tables. We use the canonical schema `(model_id VARCHAR, model BLOB)`
that `teradataml`'s `save_byom()` helper expects, with one table
per artifact kind:

- `OPUS_BYOM.onnx_models` — one row per ONNX model.
- `OPUS_BYOM.opus_tokenizers` — one row per tokenizer.

`save_byom()` is the canonical teradataml API for this pattern:
it manages the BLOB upload directly so there is no manual cursor
wrangling, and creates the destination table on first call. If a
previous run already deployed the artifact, the existing row is
removed first (the helper itself does not overwrite). Once stored,
the binaries are durable database objects — re-runs of this
notebook reuse them at zero cost.

A small naming detail: BYOM's `ONNXSeq2Seq` operator expects the
input column on the `TokenizerTable` clause to be named
`tokenizer`. Because we store the BLOB under the canonical
`save_byom` column name `model`, we alias it on `SELECT` in the
SQL invocation a few cells down (`SELECT model AS tokenizer ...`).


In [6]:
from teradataml import (
    DataFrame as TdDataFrame,
    create_context,
    execute_sql,
    get_context,
    remove_context,
    save_byom,
)

ctx = create_context(host=TD_HOST, username=TD_USER, password=TD_PASSWORD)

td_version = execute_sql(
    "SELECT InfoData FROM DBC.DBCInfoV WHERE InfoKey = 'VERSION'"
).fetchone()
print(f"Connected to Teradata {td_version[0]} as {TD_USER}@{TD_HOST}")


Connected to Teradata 20.00.29.72 as <your-user>@<your-teradata-host>


In [7]:
# Deploy the model and tokenizer with teradataml's save_byom helper.
# Both artifacts go into tables with the canonical (model_id, model)
# schema that save_byom expects — one row per artifact, BLOB upload
# managed by the helper. save_byom does not overwrite an existing
# row, so we DELETE any previous row of the same id first.
TOKENIZER_TABLE = "opus_tokenizers"

# --- model -----------------------------------------------------------
execute_sql(
    f"DELETE FROM {TD_BYOM_DATABASE}.onnx_models "
    f"WHERE model_id = '{BYOM_MODEL_ID}'"
)
print(f"Saving ONNX model to {TD_BYOM_DATABASE}.onnx_models ...")
save_byom(
    model_id=BYOM_MODEL_ID,
    model_file=str(ONNX_PATH),
    table_name="onnx_models",
    schema_name=TD_BYOM_DATABASE,
)
print("  loaded.")

# --- tokenizer -------------------------------------------------------
# First call creates OPUS_BYOM.opus_tokenizers with (model_id, model);
# subsequent calls just append after the DELETE above.
try:
    execute_sql(
        f"DELETE FROM {TD_BYOM_DATABASE}.{TOKENIZER_TABLE} "
        f"WHERE model_id = '{BYOM_MODEL_ID}'"
    )
except Exception as exc:
    if "[Error 3807]" not in str(exc):  # 3807 = "object does not exist"
        raise

print(f"Saving tokenizer to {TD_BYOM_DATABASE}.{TOKENIZER_TABLE} ...")
save_byom(
    model_id=BYOM_MODEL_ID,
    model_file=str(TOKENIZER_PATH),
    table_name=TOKENIZER_TABLE,
    schema_name=TD_BYOM_DATABASE,
)
print("  loaded.")


Saving ONNX model to OPUS_BYOM.onnx_models ...


Model is saved.
  loaded.
Saving tokenizer to OPUS_BYOM.opus_tokenizers ...
Created the model table 'opus_tokenizers' as it does not exist.


Model is saved.
  loaded.


## Step 6 — The BYOM translation query

This is the centerpiece. A single SQL `SELECT` invokes the
`TD_MLDB.ONNXSeq2Seq` table operator, which loads the deployed
model and tokenizer, encodes each input row, runs beam-search
generation inside the database, and returns the decoded English
text. The same query would translate ten million rows by changing
only the input table.

The operator takes three input streams and a set of named
parameters. We build the SQL as a plain Python string so you can
read every clause before it runs:

- **`Accumulate('id')`** — columns from the input table that pass
  straight through to the output. Here the row id, so each
  translation is matched to its source.
- **`ModelOutputTensor('sequences')`** — which output tensor of the
  ONNX graph is returned. `sequences` is the decoded token-id
  tensor; BYOM converts it to text via the tokenizer.
- **`SkipSpecialTokens('true')`** — drop `<pad>`, `</s>` and
  similar special tokens from the decoded output, so you get
  clean target-language strings.
- **`OutputLength(1024)`** — maximum bytes per translated string
  in the result column.
- **`EnableMemoryCheck('false')`** — bypass the runtime guard that
  reserves memory ahead of model load. For large models like this
  one we run with the check disabled.
- **`OverwriteCachedModel('*')`** — forces a fresh load of the
  model BLOB on this query, ensuring you always score against the
  freshest deployed binary rather than a cached copy.
- **`Const_*` cluster** — generation hyperparameters, mapped
  one-to-one onto the standard HuggingFace `model.generate(...)`
  arguments:
  - `Const_min_length(1)` / `Const_max_length(64)` — bounds on
    the number of generated tokens per row.
  - `Const_num_beams(4)` — beam-search width. Wider beams trade
    inference time for translation quality.
  - `Const_length_penalty(1.000000)` — neutral preference between
    short and long outputs.
  - `Const_repetition_penalty(1.000000)` — neutral; raise above 1
    to discourage repeated tokens.

Note that `num_return_sequences` is **not** part of the
`Const_*` cluster: it is baked into the produced ONNX graph
as a `Constant(1)` node, so each input row always returns
exactly one translation. Any `Const_num_return_sequences(N)`
USING clause is silently ignored by BYOM.


In [8]:
# Build the SQL as a multi-line string so the customer can read
# every parameter. This is the exact statement the database
# executes — nothing hidden behind a wrapper.
#
# Note the `model AS tokenizer` alias on the TokenizerTable input:
# we store the tokenizer BLOB under save_byom's canonical column
# name `model`, but the ONNXSeq2Seq operator expects the column on
# its TokenizerTable input to be named `tokenizer` — so we rename
# it inline in the SELECT.
ONNXSEQ2SEQ_SQL = f"""\
SELECT id, sequences
FROM TD_MLDB.ONNXSeq2Seq(
    ON (SELECT id, txt FROM {TD_BYOM_DATABASE}.demo_inputs_de_en) AS InputTable
    ON (SELECT model_id, model
        FROM {TD_BYOM_DATABASE}.onnx_models
        WHERE model_id = '{BYOM_MODEL_ID}') AS ModelTable DIMENSION
    ON (SELECT model AS tokenizer
        FROM {TD_BYOM_DATABASE}.{TOKENIZER_TABLE}
        WHERE model_id = '{BYOM_MODEL_ID}') AS TokenizerTable DIMENSION
    USING
        Accumulate('id')
        ModelOutputTensor('sequences')
        SkipSpecialTokens('true')
        OutputLength(1024)
        EnableMemoryCheck('false')
        OverwriteCachedModel('*')
        Const_min_length(1)
        Const_max_length(64)
        Const_num_beams(4)
        Const_length_penalty(1.000000)
        Const_repetition_penalty(1.000000)
) AS t
"""

print(ONNXSEQ2SEQ_SQL)


SELECT id, sequences
FROM TD_MLDB.ONNXSeq2Seq(
    ON (SELECT id, txt FROM OPUS_BYOM.demo_inputs_de_en) AS InputTable
    ON (SELECT model_id, model
        FROM OPUS_BYOM.onnx_models
        WHERE model_id = 'opus-mt-de-en') AS ModelTable DIMENSION
    ON (SELECT model AS tokenizer
        FROM OPUS_BYOM.opus_tokenizers
        WHERE model_id = 'opus-mt-de-en') AS TokenizerTable DIMENSION
    USING
        Accumulate('id')
        ModelOutputTensor('sequences')
        SkipSpecialTokens('true')
        OutputLength(1024)
        EnableMemoryCheck('false')
        OverwriteCachedModel('*')
        Const_min_length(1)
        Const_max_length(64)
        Const_num_beams(4)
        Const_length_penalty(1.000000)
        Const_repetition_penalty(1.000000)
) AS t



The input table `OPUS_BYOM.demo_inputs_de_en` holds the rows the
operator translates. The cell below recreates it from the demo
sentences and then executes the SQL above.


In [9]:
# Recreate the input table with this run's demo sentences via
# teradataml's copy_to_sql, which handles the CREATE TABLE + INSERT
# in one call. if_exists='replace' makes the cell idempotent.
from teradataml import copy_to_sql

DEMO_INPUT_TABLE = "demo_inputs_de_en"
DEMO_INPUT_QUALIFIED = f"{TD_BYOM_DATABASE}.{DEMO_INPUT_TABLE}"

copy_to_sql(
    df=inputs_df.rename(columns={"source_de": "txt"}),
    table_name=DEMO_INPUT_TABLE,
    schema_name=TD_BYOM_DATABASE,
    primary_index="id",
    if_exists="replace",
)
print(f"Loaded {len(inputs_df)} rows into {DEMO_INPUT_QUALIFIED}")

# Execute the BYOM query. DataFrame.from_query lets teradataml
# materialise the result lazily and render it inline as a familiar
# tabular DataFrame — no manual cursor, no fetchall, no list-to-
# pandas conversion.
print("\nExecuting ONNXSeq2Seq ...")
t_td_start = time.perf_counter()
results = TdDataFrame.from_query(ONNXSEQ2SEQ_SQL)
results_pdf = results.to_pandas(all_rows=True)
td_elapsed = time.perf_counter() - t_td_start
print(f"Returned {len(results_pdf)} rows in {td_elapsed:.2f}s "
      f"({td_elapsed / max(len(results_pdf), 1):.2f}s per row)")


Loaded 6 rows into OPUS_BYOM.demo_inputs_de_en

Executing ONNXSeq2Seq ...


Returned 6 rows in 5.46s (0.91s per row)


## Step 7 — The results

The translations as they came back from Teradata.


In [10]:
# Decode any bytes payloads from the ONNXSeq2Seq result column
# and lay the source German alongside its English translation.
def _to_str(v: object) -> str:
    return v.decode("utf-8") if isinstance(v, (bytes, bytearray)) else str(v)

td_text: dict[str, str] = {
    _to_str(row["id"]): _to_str(row["sequences"])
    for _, row in results_pdf.iterrows()
}

display_df = pd.DataFrame(
    [(rid, src, td_text.get(rid, "<MISSING>")) for rid, src in DEMO_SENTENCES],
    columns=["id", "source_de", "teradata_en"],
)
pd.set_option("display.max_colwidth", 200)
display_df


,id,source_de,teradata_en
0,greeting,"Hallo Welt, wie geht es dir heute?","Hello world, how are you today?"
1,business,Die Quartalszahlen werden nächste Woche um neun Uhr veröffentlicht.,The quarterly figures will be published at nine o'clock next week.
2,technical,Der neue Compiler optimiert den Maschinencode für moderne Prozessoren.,The new compiler optimizes the machine code for modern processors.
3,idiom,Ich verstehe nur Bahnhof.,I only understand the station.
4,entity,Albert Einstein wurde 1879 in Ulm geboren und entwickelte die Relativitätstheorie.,Albert Einstein was born in Ulm in 1879 and developed the theory of relativity.
5,long,"Obwohl es stark regnete, beschlossen wir, den Spaziergang im Park fortzusetzen, weil das Wetter laut Vorhersage am Nachmittag besser werden sollte.","Although it was raining heavily, we decided to continue the walk in the park because the weather should improve in the afternoon according to the forecast."


Note the **idiom** row. `Ich verstehe nur Bahnhof.` is a German
expression literally meaning "I only understand train station" and
idiomatically meaning "It's all Greek to me." The model produces a
faithful literal translation — a useful reminder that statistical
translation captures syntax and vocabulary cleanly but does not
always rewrite idioms into target-language equivalents. This is
behaviour of the underlying OPUS-MT checkpoint, not of BYOM.


## Step 8 — Side-by-side with HuggingFace `transformers`

A natural follow-up question: does the in-database translation
match what HuggingFace `transformers` produces locally on the
same model? The cell below translates the same six sentences via
`MarianMTModel.generate()` and lays the two outputs side by side.


In [11]:
GEN_KWARGS = dict(
    num_beams=4,
    min_length=1,
    max_length=64,
    num_return_sequences=1,
    length_penalty=1.0,
    repetition_penalty=1.0,
    early_stopping=True,
)

hf_text: dict[str, str] = {}
for rid, src in DEMO_SENTENCES:
    enc = hf_tokenizer(src, return_tensors="pt")
    out_ids = hf_model.generate(**enc, **GEN_KWARGS)
    hf_text[rid] = hf_tokenizer.decode(out_ids[0], skip_special_tokens=True)

compare_df = pd.DataFrame(
    [
        (rid, src, hf_text[rid], td_text.get(rid, "<MISSING>"),
         hf_text[rid] == td_text.get(rid, ""))
        for rid, src in DEMO_SENTENCES
    ],
    columns=["id", "source_de", "transformers_en", "teradata_en", "match"],
)
compare_df


,id,source_de,transformers_en,teradata_en,match
0,greeting,"Hallo Welt, wie geht es dir heute?","Hello world, how are you today?","Hello world, how are you today?",True
1,business,Die Quartalszahlen werden nächste Woche um neun Uhr veröffentlicht.,The quarterly figures will be published at nine o'clock next week.,The quarterly figures will be published at nine o'clock next week.,True
2,technical,Der neue Compiler optimiert den Maschinencode für moderne Prozessoren.,The new compiler optimizes the machine code for modern processors.,The new compiler optimizes the machine code for modern processors.,True
3,idiom,Ich verstehe nur Bahnhof.,I only understand the station.,I only understand the station.,True
4,entity,Albert Einstein wurde 1879 in Ulm geboren und entwickelte die Relativitätstheorie.,Albert Einstein was born in Ulm in 1879 and developed the theory of relativity.,Albert Einstein was born in Ulm in 1879 and developed the theory of relativity.,True
5,long,"Obwohl es stark regnete, beschlossen wir, den Spaziergang im Park fortzusetzen, weil das Wetter laut Vorhersage am Nachmittag besser werden sollte.","Although it was raining heavily, we decided to continue the walk in the park because the weather should improve in the afternoon according to the forecast.","Although it was raining heavily, we decided to continue the walk in the park because the weather should improve in the afternoon according to the forecast.",True


The two paths produce **identical** translations for every row.
Same model, same generation parameters, same output — the only
difference is where the inference runs.


## Cleanup

Drop the demo input table and remove the BYOM model and tokenizer
rows so the database is left in the same state it was in before
this notebook ran. The locally cached `~/.cache/teradata-opus-translate/`
files are left in place — they take ~743 MB and let you re-run
this notebook without re-exporting the ONNX model. Delete them
manually if you want a fully clean slate.


In [14]:
# --- delete the BYOM model and tokenizer rows ----------------------
execute_sql(
    f"DELETE FROM {TD_BYOM_DATABASE}.onnx_models "
    f"WHERE model_id = '{BYOM_MODEL_ID}'"
)
execute_sql(
    f"DELETE FROM {TD_BYOM_DATABASE}.{TOKENIZER_TABLE} "
    f"WHERE model_id = '{BYOM_MODEL_ID}'"
)
print(
    f"Removed BYOM rows for {BYOM_MODEL_ID!r} from "
    f"{TD_BYOM_DATABASE}.onnx_models and "
    f"{TD_BYOM_DATABASE}.{TOKENIZER_TABLE}"
)

# --- drop the demo input table -------------------------------------
execute_sql(f"DROP TABLE {DEMO_INPUT_QUALIFIED}")
print(f"Dropped {DEMO_INPUT_QUALIFIED}")

# --- close the Teradata session ------------------------------------
remove_context()
print("Teradata session closed.")


Removed BYOM rows for 'opus-mt-de-en' from OPUS_BYOM.onnx_models and OPUS_BYOM.opus_tokenizers
Dropped OPUS_BYOM.demo_inputs_de_en
Teradata session closed.


## What this demonstrated

- An open-source HuggingFace MarianMT model exported to ONNX with
  embedded beam search.
- Both model and tokenizer deployed as BLOBs inside a Teradata
  database.
- Six German sentences translated to English with a **single SQL
  query** running entirely inside Teradata — no Python sidecar,
  no data movement.
- Bit-for-bit identical output to the reference HuggingFace
  `transformers` implementation.

The pattern generalizes immediately. Any sequence-to-sequence model
that exports to ONNX with embedded beam search — translation across
the rest of the OPUS-MT family, abstractive summarization, code
generation, instruction-following — can ship this way. **Translate
at the warehouse, not in a sidecar.**
